# 12.12 decision analysis

This notebook makes the decision auditable. Facts are labelled with their source; assumptions are listed explicitly. All money is SGD unless stated otherwise.

## 1. Problem framing: facts, symptoms, and the decision

**Sources:** `data_extra/docs/swiftlane_scheduleB_and_finance_note.pdf` and the six files in `data_extra/support_calls/`.

- The decision is whether and in what format Swiftlane should run the 12.12 promotion; Marketing wants to repeat 11.11, while the CFO asks whether 11.11 made money by category.
- The decision and expected order volume must be given to the fulfilment lead by **5 December 2025** so surge capacity can be booked; Schedule B sets standard capacity at 1,200 dispatched orders/day and booked-surge capacity at 2,000/day.
- The CFO?s financial question is category economics: discount depth versus unit cost, gateway fees, fulfilment, marketing spend, and late-delivery vouchers.
- Late deliveries, complaints, refunds, and the payment/refund backlog are **symptoms**, not the underlying decision problem; they reveal the capacity and service consequences of demand.
- The underlying problems are **discount depth versus margin** and **volume versus logistics capacity** (plus a bad Kite Smartwatch E59 batch that must not be repeated).

**Say to the room:** slide 18, *?make sure it?s a problem and not just a symptom.?*

## 2. Criteria and weights (human judgement)

| Criterion | Weight / 10 | Evidence files |
|---|---:|---|
| Profit | 9 | orders, order_items, products, product_costs, finance memo in PDF |
| Customer experience | 8 | reviews.csv, support-call transcripts, ops dashboard image |
| Operational risk | 7 | Schedule B in PDF, ops dashboard image, ops voice memo |
| Competitive position | 6 | competitor flyer images and customer calls |
| Revenue | 5 | orders and order_items CSVs, competitor flyer images |
| Loyalty fairness | 4 | user_profiles.csv, platinum call, Kedai Prime flyer |

These weights are presenter judgement, not facts. The room can edit them live before accepting the recommendation.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA = ROOT / "data"
EXTRA = ROOT / "data_extra"
FLASH = pd.Timestamp("2025-11-11")
GATEWAY, FULFIL, MARKETING, VOUCHERS = 0.022, 4.20, 85_000, 1_630 * 5
BAD_SKU = 113

def load_csv(path, **kwargs):
    df = pd.read_csv(path, **kwargs)
    print(f"Loaded {path.name}: {len(df):,} rows")
    return df

o = load_csv(DATA / "orders.csv")
o["order_date"] = pd.to_datetime(o["order_date"], dayfirst=True)
i = load_csv(DATA / "order_items.csv")
p = load_csv(DATA / "products.csv")
u = load_csv(DATA / "users.csv")
u["signup_date"] = pd.to_datetime(u["signup_date"], dayfirst=True)
c = load_csv(EXTRA / "product_costs.csv")
r = load_csv(EXTRA / "reviews.csv", parse_dates=["review_date"])

f = i.merge(o, on="order_id", how="inner"); print(f"After order_items + orders join: {len(f):,} rows")
f = f.merge(p, on="product_id", how="inner"); print(f"After + products join: {len(f):,} rows")
f = f.merge(c, on="product_id", how="inner"); print(f"After + product_costs join: {len(f):,} rows")
f["revenue"] = f.quantity * f.unit_price
f["cogs"] = f.quantity * f.unit_cost_sgd
f["discount"] = 1 - f.unit_price / f.list_price
print(f"Total revenue across all orders/statuses (check): S${f.revenue.sum():,.2f}")


In [ ]:
# 5a. Delivered-only 11.11 P&L. Refunded and cancelled orders are excluded because they are not delivered.
day = f[f.order_date.eq(FLASH)].copy(); print(f"11.11 filter: {len(day):,} line rows")
d = day[day.status.eq("delivered")].copy(); print(f"Delivered-only filter: {len(d):,} line rows; statuses included: {d.status.unique().tolist()}")
d.loc[d.product_id.eq(BAD_SKU), "revenue"] = 0  # finance memo: E59 recalled/refunded, stock written off
d["fulfilment"] = FULFIL * d.revenue / d.groupby("order_id").revenue.transform("sum").replace(0, 1)
cat = d.groupby("category").agg(revenue=("revenue","sum"), cost_of_goods=("cogs","sum"), fulfilment=("fulfilment","sum"))
cat["gross_margin"] = cat.revenue - cat.cost_of_goods
cat["payment_gateway_fee"] = cat.revenue * GATEWAY
cat["contribution"] = cat.gross_margin - cat.payment_gateway_fee - cat.fulfilment
cat["contribution_margin_%"] = 100 * cat.contribution / cat.revenue
print(cat.sort_values("contribution").round(0).to_string())
print(f"\nContribution before campaign costs: S${cat.contribution.sum():,.0f}")
print(f"Marketing spend: S${MARKETING:,.0f}; late vouchers: 1,630 x S$5 = S${VOUCHERS:,.0f}")
print(f"11.11 result after campaign costs: S${cat.contribution.sum() - MARKETING - VOUCHERS:,.0f}")
e59 = day[day.product_id.eq(BAD_SKU) & day.status.eq("delivered")]
print(f"E59 check: {e59.quantity.sum():.0f} units; revenue refunded S${e59.revenue.sum():,.0f}; stock written off S${e59.cogs.sum():,.0f}")


In [ ]:
# 5b. Review quality and multilingual complaint analysis. The text rule is a transparent proxy, not a census of all customers.
print(f"Review rows: {len(r):,}; exact duplicates: {r.duplicated().sum()}; orphan order IDs: {(~r.order_id.isin(o.order_id)).sum()}; ratings outside 1-5: {(~r.rating.between(1,5)).sum()}")
rc = r.drop_duplicates(); print(f"After duplicate removal: {len(rc):,} rows")
rc = rc[rc.order_id.isin(o.order_id) & rc.rating.between(1,5)]; print(f"After orphan/bad-rating removal: {len(rc):,} rows")
rc = rc.merge(o[["order_id","order_date"]], on="order_id", how="inner"); print(f"After reviews + orders join: {len(rc):,} rows")
rc = rc.merge(p[["product_id","category"]], on="product_id", how="inner"); print(f"After + products join: {len(rc):,} rows")
rc["flash"] = rc.order_date.eq(FLASH)
rc["delivery_complaint"] = rc.text.str.contains(r"days|late|lambat|lewat|\u6162|delivery", case=False, na=False) & rc.rating.le(3)
print(rc.groupby("flash").agg(reviews=("rating","size"), average_rating=("rating","mean"), delivery_complaint_share=("delivery_complaint","mean")).round(2).to_string())
print("11.11 delivery-complaint share by category (%):")
print((rc[rc.flash].groupby("category").delivery_complaint.mean()*100).round(0).sort_values(ascending=False).to_string())
print("Languages in clean data:", rc.language.value_counts().to_dict())
w = rc[rc.product_id.eq(BAD_SKU)].copy(); hot = w.text.str.contains(r"hot|overheat|swell|warm|panas|\u70eb|battery", case=False, na=False); batch = w.order_date.between("2025-11-11","2025-12-31")
print(f"Kite Smartwatch E59 safety cluster: {hot.sum()} of {len(w)} reviews mention heat/battery; batch window {hot[batch].sum()} of {batch.sum()}, other dates {hot[~batch].sum()} of {(~batch).sum()}")


## 4. Competitor flyers and alternatives

The two flyer images are `data_extra/images/competitor_shopnova_1212.png` and `data_extra/images/competitor_kedaiprime_1212.png`. The image evidence transcribes as follows:

| Competitor | Categories | Offer | Dates / access | Currency | Delivery promise |
|---|---|---|---|---|---|
| ShopNova | Electronics | Up to 60% off | 10?12 Dec | SGD (S$) | Free next-day delivery over S$50 |
| Kedai Prime | Beauty and Fashion | Buy 2, get 25% off; bundles from RM49 | Platinum early access 11 Dec, 8pm | RM (ringgit) | Not stated on flyer |

**Currency assumption:** for comparisons only, use RM1 = S$0.30; the flyer?s RM49 bundle is therefore about S$14.70. This rate is an assumption, not supplied by the files, and should be replaced by the finance-approved rate before launch.

Four alternatives: **A** repeat 11.11 as-is for one day; **B** targeted discounts for one day; **C** same discounts spread over three days; **D** targeted discounts spread over three days plus platinum early access.


In [ ]:
# 5c. Dashboard transcription and source cross-check. The received series is computed from orders.csv; dispatched and on-time values are transcribed from the dashboard image.
received = o.groupby("order_date").size().loc[pd.Timestamp("2025-11-09"):pd.Timestamp("2025-11-20")].rename("orders_received")
capacity = 1200
dispatched = []; backlog = 0
for date, n in received.items():
    dispatched.append(min(capacity, backlog + n)); backlog = max(0, backlog + n - capacity)
ops = pd.DataFrame({"orders_received": received.values, "orders_dispatched": dispatched, "on_time_pct": [97,96,97,93,46,44,58,79,91,95,96,97]}, index=received.index)
print(f"Dashboard dataframe rows: {len(ops)}")
print(ops.to_string())
print(f"11 Nov received: {ops.loc['2025-11-11','orders_received']:,} (expected 3,000); capacity line: {capacity:,}/day (Schedule B)")
print("Sources agree on 11 Nov received, 1,200/day standard capacity, and 44% on-time delivery on 14 Nov.")


In [ ]:
# 5d. Test the loyalty and new-customer claims.
buyers = set(day.user_id); print(f"11.11 buyers: {len(buyers):,}")
def avg_spend(start, end):
    w = f[(f.order_date.ge(start)) & (f.order_date.lt(end)) & (~f.order_date.eq(FLASH))]
    totals = w.groupby("user_id").revenue.sum()
    return totals[totals.index.isin(buyers)].sum()/len(buyers), totals[~totals.index.isin(buyers)].sum()/(u.user_id.nunique()-len(buyers))
pre = avg_spend("2025-10-01","2025-11-11"); post = avg_spend("2025-11-12","2026-02-10")
print(f"Average spend/user before: flash buyers S${pre[0]:,.0f} vs others S${pre[1]:,.0f} ({pre[0]/pre[1]:.1f}x)")
print(f"Average spend/user after: flash buyers S${post[0]:,.0f} vs others S${post[1]:,.0f} ({post[0]/post[1]:.1f}x)")
print(f"Sign-ups after 30 Sep 2025: {(u.signup_date > '2025-09-30').sum()}")
print("Conclusion: the post-sale difference is not causal evidence because flash buyers already spent more before; the new-customer claim has no support in these sign-up dates.")
print("Say to the room: slide 30, management filters. Would anyone have checked if the number agreed with what they hoped?")


In [ ]:
# 5e. Break-even discount and sensitivity.
unit_economics = f.drop_duplicates("product_id").groupby("category").apply(lambda g: (g.unit_cost_sgd/g.list_price).mean(), include_groups=False)
break_even = (1 - unit_economics/(1-GATEWAY)).mul(100).sort_values().rename("break_even_discount_pct")
print(break_even.round(1).to_string())
levels = [10,20,30,40,50]
sensitivity = pd.DataFrame({f"{x}% off": (((1-x/100)*(1-GATEWAY)-unit_economics)*100).round(1) for x in levels})
print(sensitivity.to_string())
print("Categories that should never go deeper than 25%: Electronics and Groceries (their break-even discounts are 25.6% and 28.3%).")


In [ ]:
# 6. Alternatives, weighted scoring, and sensitivity. Scores are judgement, not measurements.
weights = pd.Series({"Profit":9,"Customer experience":8,"Operational risk":7,"Competitive position":6,"Revenue":5,"Loyalty fairness":4})
scores = pd.DataFrame({"A Repeat as-is": [1,2,3,6,9,3], "B Targeted one day": [7,4,5,5,6,4], "C Same discounts / 3 days": [2,8,8,6,9,3], "D Targeted / 3 days / platinum": [8,8,8,5,6,7]}, index=weights.index)
weighted = scores.mul(weights, axis=0).sum().sort_values(ascending=False)
print("Scores:"); print(scores); print("\nWeighted totals:", weighted.to_dict())
print("Evidence basis: P&L and break-even table support Profit; review/ops tables support Customer experience and Operational risk; flyer and call evidence support Competitive position and Loyalty fairness; revenue is the scenario output, so all scores remain contestable.")
# Scenario sensitivity with Revenue fixed at 10 and Profit swept 1..10.
for profit_weight in range(1,11):
    w = weights.copy(); w["Profit"] = profit_weight; w["Revenue"] = 10
    winner = scores.mul(w, axis=0).sum().idxmax()
    print(f"Profit={profit_weight}, Revenue=10 -> {winner}")
print("Under the stated weights D wins. With Revenue=10, C wins at Profit 1-3 and D wins from about Profit 4 onward; this is a values choice, not something AI can settle.")


## 7. One-page recommendation for the CFO

**Decision:** choose **targeted discounts spread over three days, with platinum early access** (alternative D). Book the required logistics plan and give Swiftlane the decision plus expected order volume by **5 December 2025**. Exclude the Kite Smartwatch E59 batch and do not discount Electronics or Groceries deeper than 25%.

**Three numbers:** (1) 11.11 contribution after category costs but before campaign costs was about **S$90k negative**, and after **S$85k marketing + S$8,150 vouchers** the result was about **S$183k negative**; (2) Electronics alone contributed about **S$189k negative**; (3) on-time delivery fell to **44% on 14 Nov** when demand exceeded the 1,200/day standard capacity.

**Risks and mitigations:** targeted discounts may reduce volume (assumption); monitor revenue and order volume daily. Three days lowers the peak capacity risk, but surge is capped at 2,000/day. The E59 safety cluster requires exclusion and an incident/stock-control check. Platinum early access improves fairness versus the competitor promise, but must be communicated clearly.

**13 December KPIs:** contribution after marketing and vouchers; revenue and orders by category; discount rate and Electronics/Groceries margin; orders received versus dispatched and backlog; on-time delivery percentage; delivery-complaint share and average rating; refunds and voucher count; platinum early-access conversion and complaint rate.

## 8. Assumptions rather than facts from files

- The RM-to-SGD conversion rate of RM1 = S$0.30.
- A smaller targeted discount produces the scenario volume response used in the scoring (the notebook should be rerun if management chooses another response).
- Fulfilment cost is allocated to categories by each line?s share of order revenue.
- Review text plus low ratings are a proxy for customer sentiment, and the multilingual keyword rule identifies delivery complaints.
- Scenario scores from 1 to 10 and the recommendation weights are human judgement.
- The alternative scenario uses the 11.11 customer/basket pattern as a base and treats non-E59 units as available for 12.12.
